In [4]:
# =============================================================
#  BANK MARKETING PROJECT (Machine Learning)
# =============================================================

In [5]:
# ==============================
# 1. INSTALL DEPENDENCIES
# ==============================

!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy

In [6]:
# ==============================
# 2. IMPORTS LIBRARIES
# ==============================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix)

from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import ADASYN

In [7]:
# ==============================
# 3. LOAD DATASET
# ==============================

df = pd.read_csv('/content/bank-additional-full (1).csv', sep=';')

In [8]:
# ==============================
# 4. PREPROCESSING
# ==============================

# Drop Leakage Feature
df = df.drop('duration', axis=1)

# Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

# Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)

In [9]:
# ==============================
# 5. SPLIT DATA
# ==============================

# Split Features & Target
X = df.drop('y', axis=1)
y = df['y']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# ADASYN Oversampling
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_scaled, y_train)

In [10]:
# =========================================
# 6. CUSTOM METRICS: SPECIFICITY & G-MEAN
# =========================================
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def gmean(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn)
    spec = tn / (tn + fp)
    return np.sqrt(sensitivity * spec)

In [11]:
# ==============================
# 7. EVALUATION METRICS
# ==============================
from sklearn.metrics import make_scorer

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'specificity': make_scorer(specificity),
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'mcc': make_scorer(matthews_corrcoef),
    'g_mean': make_scorer(gmean),
    'kappa': make_scorer(cohen_kappa_score)
}

In [12]:
# ==============================
# 8. METRICS FUNCTION
# ==============================

def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision   : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall      : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Specificity : {tn / (tn + fp):.4f}")
    print(f"F1 Score    : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC     : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"MCC         : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"G-Mean      : {geometric_mean_score(y_true, y_pred):.4f}")
    print(f"Kappa       : {cohen_kappa_score(y_true, y_pred):.4f}")

In [13]:
# ==============================
# 9. CROSS VALIDATION
# ==============================

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [20]:
# ==============================
# MODEL: MLP
# ==============================


from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV

mlp = MLPClassifier(
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=3,
    random_state=42
)

params_mlp = {
    "hidden_layer_sizes": [(16), (32)],
    "activation": ["relu"],
    "alpha": [1.0,5.0,10.0,20.0],
    "learning_rate_init": [1e-4],
    "batch_size": [32, 64],
    "solver": ["adam"],

}

rscv_mlp = RandomizedSearchCV(
    mlp,
    param_distributions=params_mlp,
    n_iter=12,
    scoring='precision',
    cv=3,
    n_jobs=-1,
    random_state=42
)

rscv_mlp.fit(X_train_ad, y_train_ad)

best_mlp = rscv_mlp.best_estimator_

y_prob_mlp = best_mlp.predict_proba(X_test_scaled)[:, 1]
y_pred_mlp = best_mlp.predict(X_test_scaled)
print("Model: MLP")
compute_metrics(y_test, y_pred_mlp, y_prob_mlp)

Model: MLP
Accuracy    : 0.7840
Precision   : 0.3026
Recall      : 0.7026
Specificity : 0.7944
F1 Score    : 0.4230
ROC-AUC     : 0.7997
MCC         : 0.3575
G-Mean      : 0.7471
Kappa       : 0.3151


In [21]:
# ==============================
# MODEL: Bagging
# ==============================
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV


base_tree = DecisionTreeClassifier(
    max_depth=None,
    class_weight='balanced',
    random_state=42
)

bagging = BaggingClassifier(
    estimator=base_tree,
    n_estimators=500,
    n_jobs=-1,
    random_state=42
)

params_bag = {
    "max_samples": [0.8, 1.0],
    "max_features": [0.8, 0.9],
    "estimator__min_samples_split": [2, 5],
    "estimator__min_samples_leaf": [1, 2]
}

rscv_bag = RandomizedSearchCV(
    bagging,
    param_distributions=params_bag,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rscv_bag.fit(X_train_ad, y_train_ad)
best_bag = rscv_bag.best_estimator_


y_pred_bag = best_bag.predict(X_test_scaled)
y_prob_bag = best_bag.predict_proba(X_test_scaled)[:, 1]

print("Model: Bagging ")
compute_metrics(y_test, y_pred_bag, y_prob_bag)



Fitting 3 folds for each of 10 candidates, totalling 30 fits
Model: Bagging 
Accuracy    : 0.8927
Precision   : 0.5368
Recall      : 0.3459
Specificity : 0.9621
F1 Score    : 0.4207
ROC-AUC     : 0.7835
MCC         : 0.3753
G-Mean      : 0.5769
Kappa       : 0.3646
